# 📘 Fine-tuning GPT-2 for Instruction Following — Teaching Edition

> **What this notebook does (in one sentence):** Take a pre-trained GPT-2 model and teach it to follow human-style instructions (like *"Identify the correct spelling of this word"*) by training it on a small dataset of instruction → response pairs in **Alpaca format**.

---

## 🗺️ Notebook Roadmap

| Section | Cells | What happens |
|---|---|---|
| **1. Setup & Data Download** | 1–3 | Download a JSON file of 1,100 instruction-response examples |
| **2. Alpaca Formatting** | 4–6 | Wrap raw data into a standard prompt format |
| **3. Train / Val / Test Split** | 7 | Carve the data into 3 chunks |
| **4. Dataset Class** | 8–9 | Wrap the data so PyTorch can load it efficiently |
| **5. Collate Functions** | 10–15 | Pad sequences in a batch to equal length & build targets |
| **6. Cross-Entropy Loss Demo** | 16–18 | Show how the loss function and `-100` ignore-index work |
| **7. DataLoaders** | 19–22 | Create batched iterators over train/val/test |
| **8. Loading Pre-trained GPT-2** | 23–24 | Download GPT-2 weights and try the model BEFORE fine-tuning |
| **9. Initial Loss Measurement** | 25–26 | Measure the loss on the model BEFORE training |
| **10. Fine-tuning Loop** | 27 | Actually train (fine-tune) the model |
| **11. Plot Losses** | 28 | Visualize how training went |

---

## 🧠 Big Picture: What is Fine-tuning?

A pre-trained language model (like GPT-2) has already learned how to predict the next word from billions of pages of text. But it doesn't naturally know that when a human writes "### Instruction: ... ### Response: ..." it's supposed to follow the instruction. **Fine-tuning** is the process of taking that pre-trained model and continuing to train it on a much smaller, specialized dataset — here, instruction-following data — so it learns this new behavior.

We're using the **Alpaca format**, which structures every training example like this:

```
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
<the user's request>

### Input:
<optional extra context>

### Response:
<the answer the model should produce>
```

The model learns to output the `### Response:` part when it sees the rest.


---

# 🟦 SECTION 1 — Setup & Data Download

## 📦 Cell 1 — Download the instruction dataset

This cell defines a helper function that **downloads a JSON file from the internet only once** (caching it locally), then loads it into Python as a list of dictionaries.

### Line-by-line breakdown

<details>
<summary><b><code>import os</code></b></summary>

- **What it does:** Loads Python's built-in `os` module, which lets us interact with the operating system (check if files exist, list directories, etc.).
- **Syntax:** `import` is a keyword that pulls in another file/library. `os` is its name.
- **Why:** We need `os.path.exists()` later to check if the data file is already downloaded.
- **Analogy:** Like opening a toolbox before you start a project.

</details>

<details>
<summary><b><code>import urllib</code></b></summary>

- **What it does:** Loads `urllib`, Python's built-in library for working with URLs (downloading from the web).
- **Why:** We use it to fetch the JSON file from a GitHub URL.

</details>

<details>
<summary><b><code>import json</code></b></summary>

- **What it does:** Loads `json`, Python's library for reading/writing JSON (a text format for structured data — basically nested dictionaries and lists).
- **Why:** Our dataset is a `.json` file; we need this to parse it into a Python list.

</details>

<details>
<summary><b><code>def download_and_load_file(file_path, url):</code></b></summary>

- **What it does:** Defines a new function called `download_and_load_file` that takes two inputs (called *parameters*): `file_path` (where to save/read the file locally) and `url` (where to download from).
- **Syntax:** `def` = "define a function". The parentheses list the inputs. The colon `:` starts the function's body (everything indented below belongs to it).
- **Why:** Wrapping logic in a function lets us reuse it and keep the code tidy.

</details>

<details>
<summary><b><code>if not os.path.exists(file_path):</code></b></summary>

- **What it does:** Checks "does this file *not* already exist on disk?". If true, we'll download it.
- **Syntax:** `if` runs the indented block when the condition is true. `not` flips True↔False. `os.path.exists(...)` returns True if the file exists.
- **Why:** Avoid re-downloading the same file every time the cell runs.

</details>

<details>
<summary><b><code>with urllib.request.urlopen(url) as response:</code></b></summary>

- **What it does:** Opens a connection to the URL and stores the response object as the variable `response`. The `with` statement makes sure the connection is closed automatically when we're done.
- **Why:** Cleaner than manually opening and closing connections.
- **Analogy:** `with` is like "borrow this, and put it back when finished."

</details>

<details>
<summary><b><code>data = response.read().decode('utf-8')</code></b></summary>

- **What it does:** Reads the raw bytes of the response, then decodes them into a regular text string using UTF-8 encoding (the standard for web text).
- **Why:** Web data arrives as bytes (raw 1s and 0s). We need real text to save into a file.

</details>

<details>
<summary><b><code>with open(file_path, 'w', encoding='utf-8') as file:</code></b></summary>

- **What it does:** Opens a file at `file_path` in **write mode** (`'w'`) using UTF-8 encoding.
- **Why:** We need to save the downloaded text locally.

</details>

<details>
<summary><b><code>file.write(data)</code></b></summary>

- **What it does:** Writes the string `data` into the open file.

</details>

<details>
<summary><b><code>else:</code> &nbsp; <code>with open(file_path, 'r', encoding='utf-8') as file:</code> &nbsp; <code>text_data = file.read()</code></b></summary>

- **What it does:** If the file *did* already exist, just open it in **read mode** (`'r'`) and read its contents into `text_data`.
- ⚠️ **Note:** `text_data` here is unused — it's a slight redundancy (the actual loading is done below with `json.load`). It doesn't break anything.

</details>

<details>
<summary><b><code>with open(file_path, 'r', encoding='utf-8') as file:</code> &nbsp; <code>data = json.load(file)</code></b></summary>

- **What it does:** Opens the file again in read mode, and uses `json.load()` to parse the JSON text into a Python object (here, a list of dictionaries).
- **Why:** `json.load()` understands the JSON format and converts `{...}` text into real Python `dict` objects.

</details>

<details>
<summary><b><code>return data</code></b></summary>

- **What it does:** Sends the parsed list back to whoever called the function.

</details>

<details>
<summary><b><code>file_path = "instruction-data.json"</code></b></summary>

- **What it does:** Stores the local filename as a string in the variable `file_path`.

</details>

<details>
<summary><b><code>url = "https://raw.githubusercontent.com/..."</code></b></summary>

- **What it does:** The full web address of the JSON file.

</details>

<details>
<summary><b><code>data = download_and_load_file(file_path, url)</code></b></summary>

- **What it does:** Calls the function, downloads (or loads) the file, and stores the result in `data` — a list of dictionaries.

</details>

<details>
<summary><b><code>print("Number of entries:", len(data))</code></b></summary>

- **What it does:** Prints how many entries are in the list. `len(...)` returns the number of items.
- **Output:** `Number of entries: 1100`

</details>

### 🧪 Real-world analogy
Like a librarian who first checks if the book is on the shelf — if yes, just hand it over; if not, order it from the warehouse, then hand it over.

In [ ]:
import os
import urllib
import json

def download_and_load_file(file_path, url):
    # Only download if the file isn't already on disk (saves bandwidth)
    if not os.path.exists(file_path):
        # Open a connection to the remote URL
        with urllib.request.urlopen(url) as response:
            # Read raw bytes -> decode to a UTF-8 text string
            data = response.read().decode('utf-8')

        # Save the downloaded text to disk so we don't re-download next time
        with open(file_path, 'w', encoding='utf-8') as file:
            file.write(data)

    else:
        # If file already exists, just open it (we'll re-load via json.load below)
        with open(file_path, 'r', encoding='utf-8') as file:
            text_data = file.read()  # (note: this variable is unused)

    # Parse the JSON file into a Python list of dicts
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)

    return data

# The local filename we'll save the data as
file_path = "instruction-data.json"
# Source URL: Sebastian Raschka's "LLMs from scratch" repo, chapter 7 dataset
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json"

# Download (or load from disk) and store as a list of dicts
data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))   # Expected: 1100


## 📦 Cell 2 — Peek at one example

### 🔹 Line of Code
```python
print("Example entry:\n", data[50])
```

### 🧠 What it does
Prints the **51st item** in the dataset (Python uses 0-based indexing — `data[0]` is the first item, `data[50]` is the 51st).

### ⚙️ Syntax breakdown
- `print(...)` — built-in function that displays text in the output cell.
- `"Example entry:\n"` — a string. `\n` is a special character that means *newline* (line break).
- `,` — separates the two arguments to `print`. `print` adds a space between them.
- `data[50]` — square brackets `[ ]` index into a list to grab item #50.

### 🤔 Why this is used
Just a sanity check so we can *see* what one entry actually looks like.

### 📦 Output
```
Example entry:
 {'instruction': 'Identify the correct spelling of the following word.',
  'input': 'Ocassion',
  'output': "The correct spelling is 'Occasion.'"}
```

### 🧪 Real-world analogy
Like opening a random page of a textbook to glance at the format before reading the whole thing.

In [ ]:
# Inspect a single example to understand the data structure.
# Each entry is a dict with three keys: 'instruction', 'input', 'output'.
print("Example entry:\n", data[50])


## 📦 Cell 3 — Another example (showing empty input)

### 🔹 Line of Code
```python
print("Another example entry:\n", data[999])
```

### 🧠 What it does
Prints item #1000. This one has an **empty `input` field** — many entries don't need extra input beyond the instruction itself. We need to handle both cases later.

### 📦 Output
```
Another example entry:
 {'instruction': "What is an antonym of 'complicated'?",
  'input': '',
  'output': "An antonym of 'complicated' is 'simple'."}
```

### 🔗 Connection
This is why the next cell's `format_input` function has an `if entry["input"]` check — to skip the `### Input:` block when input is empty.

In [ ]:
# Look at an entry where 'input' is an empty string ''.
# Many instructions don't need extra context; our formatter must handle this.
print("Another example entry:\n", data[999])


---

# 🟩 SECTION 2 — Alpaca Prompt Formatting

We now define the function that turns each raw `{instruction, input, output}` dict into the **Alpaca-style prompt string** that the model will see during training.

## 📦 Cell 4 — `format_input` function

### Line-by-line

<details>
<summary><b><code>def format_input(entry):</code></b></summary>

- Defines a function that takes one parameter `entry` (one dict from our dataset).

</details>

<details>
<summary><b><code>instruction_text = (
    f"Below is an instruction that describes a task. "
    f"Write a response that appropriately completes the request."
    f"\n\n### Instruction:\n{entry['instruction']}"
)</code></b></summary>

- **What it does:** Builds a multi-line **f-string** (formatted string) in three parts that get concatenated together because Python automatically joins adjacent string literals.
- **Syntax:**
  - `f"..."` is an *f-string*. Anything inside `{...}` is a Python expression that gets evaluated and inserted.
  - `\n` = newline character.
  - `entry['instruction']` reads the `'instruction'` value from the dict.
- **Result:** A string like:
  ```
  Below is an instruction that describes a task. Write a response that appropriately completes the request.

  ### Instruction:
  Identify the correct spelling of the following word.
  ```

</details>

<details>
<summary><b><code>input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""</code></b></summary>

- **What it does:** A **conditional expression** (the "ternary" form). Reads:
  > IF `entry["input"]` is truthy (non-empty), set `input_text` to the formatted "### Input:" block. ELSE, set it to an empty string.
- **Why:** Empty strings are *falsy* in Python — so an empty `input` field skips the whole `### Input:` section.

</details>

<details>
<summary><b><code>return instruction_text + input_text</code></b></summary>

- Concatenates the two pieces and returns the final prompt string.

</details>

### 🧪 Real-world analogy
Like a form letter template: you fill in the blanks, but skip the "P.S." section if there's nothing to add.

In [ ]:
def format_input(entry):
    """Wrap a {instruction, input, output} dict into the Alpaca prompt format."""

    # Always include the system header + instruction
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    # Only add the '### Input:' block if 'input' is non-empty
    # (empty strings are 'falsy' in Python)
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    # Concatenate and return
    return instruction_text + input_text


## 📦 Cell 5 — Show a full Alpaca example (with response)

### Lines:
```python
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + desired_response)
```

### What each line does

<details>
<summary><b><code>model_input = format_input(data[50])</code></b></summary>

- Calls our function with the 51st entry; gets back the prompt string (instruction + maybe input). Stores it.

</details>

<details>
<summary><b><code>desired_response = f"\n\n### Response:\n{data[50]['output']}"</code></b></summary>

- Builds the `### Response:` portion separately. This is what the model is supposed to learn to produce.

</details>

<details>
<summary><b><code>print(model_input + desired_response)</code></b></summary>

- Concatenates everything (input + target response) and prints the **complete Alpaca-formatted training example**.

</details>

### 🤔 Why split it into `model_input` and `desired_response`?
At **inference time** (when actually using the model), we'll only feed in `model_input` and ask the model to generate the response. But during **training**, we feed the full thing so the model learns to produce the response part.

In [ ]:
# Build a full training example by hand for entry #50
model_input = format_input(data[50])

# The 'desired_response' is what we want the model to learn to output
desired_response = f"\n\n### Response:\n{data[50]['output']}"

# Print the FULL Alpaca-formatted example
# (during training the model sees this entire string)
print(model_input + desired_response)  # alpaca format used for fine-tuning


## 📦 Cell 6 — Same thing for an entry with no `input`

This shows the format **without** the `### Input:` block (because `data[999]['input']` is empty).

In [ ]:
# Same demonstration on entry #999, which has an empty 'input' field.
# Notice the '### Input:' section is omitted automatically.
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"
print(model_input + desired_response)


---

# 🟨 SECTION 3 — Train / Validation / Test Split

We split the 1,100 examples into three sets:
- **Training** (85% = 935 examples) — the data the model actually learns from.
- **Validation** (5% = 55 examples) — used during training to monitor overfitting.
- **Test** (10% = 110 examples) — held out for final evaluation.

## 📦 Cell 7 — Splitting the data

### Line-by-line

<details>
<summary><b><code>train_end = int(0.85 * len(data))</code></b></summary>

- **What:** Compute the index where training ends. `len(data) = 1100`, so `0.85 * 1100 = 935.0`, and `int(935.0) = 935`.
- **Why `int(...)`:** array indices must be whole numbers.

</details>

<details>
<summary><b><code>test_end = train_end + int(0.1 * len(data))</code></b></summary>

- `935 + int(110.0) = 935 + 110 = 1045`. Test occupies indices 935..1044.

</details>

<details>
<summary><b><code>val_end = test_end + int(0.05 * len(data))</code></b></summary>

- `1045 + 55 = 1100`. Validation occupies indices 1045..1099.

</details>

<details>
<summary><b><code>train_data = data[:train_end]</code></b></summary>

- **Slicing:** `data[:935]` returns items 0 through 934 (the colon `:` means "from start to end-index, exclusive").

</details>

<details>
<summary><b><code>test_data = data[train_end:test_end]</code></b></summary>

- `data[935:1045]` — items 935..1044.

</details>

<details>
<summary><b><code>val_data = data[test_end:val_end]</code></b></summary>

- `data[1045:1100]` — items 1045..1099.

</details>

### ⚠️ Subtle observation
Notice the order is `train → test → val` in the slicing, but printed as `train → val → test`. The validation set sits at the *end* of the file, not in the middle. As long as the data isn't sorted by topic this is fine, but it's slightly unusual.

### 🧪 Real-world analogy
You have a deck of 100 flashcards. You give 85 to a student to study (train), keep 5 to quiz them during studying (val), and save 10 for the final exam (test).

In [ ]:
# --- Compute split sizes ---
train_end = int(0.85 * len(data))                  # 935
test_end  = train_end + int(0.1 * len(data))       # 935 + 110 = 1045
val_end   = test_end  + int(0.05 * len(data))      # 1045 + 55 = 1100

# --- Apply slicing to chop the list into 3 disjoint pieces ---
train_data = data[:train_end]            # indices [0,   935)
test_data  = data[train_end:test_end]    # indices [935, 1045)
val_data   = data[test_end:val_end]      # indices [1045,1100)

print("Training set length:", len(train_data))     # 935
print("Validation set length:", len(val_data))     # 55
print("Test set length:", len(test_data))          # 110


---

# 🟧 SECTION 4 — PyTorch Dataset & Tokenizer

## 🧠 PyTorch concept: `Dataset`

PyTorch trains models on data using two key abstractions:

1. **`Dataset`** — knows *how to fetch one example by index*.
2. **`DataLoader`** — knows *how to bundle examples into batches and iterate over them*.

To make a custom Dataset, you subclass `torch.utils.data.Dataset` and implement two methods:
- `__len__(self)` — how many examples total?
- `__getitem__(self, index)` — give me example #index.

PyTorch then handles batching, shuffling, and parallel loading for you.

---

## 📦 Cell 8 — `InstructionDataset` class

### Line-by-line

<details>
<summary><b><code>import torch</code></b></summary>

- Loads the PyTorch library, the deep-learning framework we use throughout.

</details>

<details>
<summary><b><code>from torch.utils.data import Dataset</code></b></summary>

- Imports just the `Dataset` class from PyTorch's data utilities. Saves us typing `torch.utils.data.Dataset`.

</details>

<details>
<summary><b><code>class InstructionDataset(Dataset):</code></b></summary>

- Defines a new class called `InstructionDataset` that **inherits from** `Dataset`. The parentheses `(Dataset)` means "this is a special kind of Dataset."

</details>

<details>
<summary><b><code>def __init__(self, data, tokenizer):</code></b></summary>

- The **constructor** — called when you do `InstructionDataset(...)`. Takes `data` (the list of dicts) and `tokenizer` (the GPT-2 tokenizer that turns text → integer IDs).
- `self` is the conventional first parameter referring to the new instance being built.

</details>

<details>
<summary><b><code>self.encoded_texts = []</code></b></summary>

- Initializes an empty list **on this instance**. We'll fill it with token-ID lists.

</details>

<details>
<summary><b><code>self.data = data</code></b></summary>

- Stores the raw data on the instance for later access.

</details>

<details>
<summary><b><code>for entry in data:</code></b></summary>

- Loops through every example in the dataset.

</details>

<details>
<summary><b><code>formatted_input = format_input(entry)</code></b><br>
<code>response_text = f"\n\n### Response:\n{entry['output']}"</code><br>
<code>full_text = formatted_input + response_text</code></summary>

- Builds the full Alpaca-formatted string (instruction + maybe input + response).

</details>

<details>
<summary><b><code>self.encoded_texts.append(tokenizer.encode(full_text))</code></b></summary>

- **`tokenizer.encode(text)`** turns the human-readable string into a **list of integer token IDs** (e.g., `[12, 405, 8, ...]`). This is what the model actually sees.
- We append that list to `self.encoded_texts`. So `self.encoded_texts[i]` is a list of ints for example #i.
- 💡 We **pre-encode all examples once at construction time** rather than re-tokenizing on every fetch — this makes data loading much faster during training.

</details>

<details>
<summary><b><code>def __getitem__(self, index): return self.encoded_texts[index]</code></b></summary>

- PyTorch will call this when it needs example #index. We return the pre-tokenized list of ints.

</details>

<details>
<summary><b><code>def __len__(self): return len(self.data)</code></b></summary>

- PyTorch calls this to know how many examples there are.

</details>

### 🧪 Real-world analogy
A library catalog: `__len__` tells you how many books exist; `__getitem__` retrieves the book at slot N. We pre-tokenize so the books are already translated to a common language.

In [ ]:
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    """
    A PyTorch Dataset that pre-tokenizes every (instruction, input, output)
    triple into a single list of integer token IDs.
    """

    def __init__(self, data, tokenizer):
        # Will hold one list-of-ints per example
        self.encoded_texts = []
        self.data = data

        # Tokenize every example up-front (once) for speed during training
        for entry in data:
            # 1. Build the instruction-+-input prompt portion
            formatted_input = format_input(entry)
            # 2. Append the response portion (what the model should learn to output)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = formatted_input + response_text
            # 3. Convert text -> list of integer token IDs and store
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        # Return the i-th tokenized example (a list of ints)
        return self.encoded_texts[index]

    def __len__(self):
        # Total number of examples
        return len(self.data)


## 📦 Cell 9 — Load the GPT-2 tokenizer

### Lines
```python
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))
```

### Line-by-line

<details>
<summary><b><code>import tiktoken</code></b></summary>

- `tiktoken` is OpenAI's fast tokenizer library. It can split text into the same token IDs that GPT-2 (and GPT-3/4) use internally.

</details>

<details>
<summary><b><code>tokenizer = tiktoken.get_encoding("gpt2")</code></b></summary>

- Loads the specific encoding scheme used by GPT-2. The result is a tokenizer object with `.encode()` (text → IDs) and `.decode()` (IDs → text) methods.

</details>

<details>
<summary><b><code>print(tokenizer.encode("&lt;|endoftext|&gt;", allowed_special={"&lt;|endoftext|&gt;"}))</code></b></summary>

- **What this prints:** `[50256]`.
- `<|endoftext|>` is GPT-2's **special end-of-text token** with ID **50256**. By default, the tokenizer refuses to encode special tokens (they could be a source of injection attacks). The `allowed_special={...}` argument explicitly says "yes, encode this special one."
- **Why we care:** The number `50256` will be reused throughout this notebook as our **padding token** (i.e., filler used to make all sequences in a batch the same length).

</details>

### 🧪 Real-world analogy
Think of the tokenizer as a translator that converts English into a numbered language only the model understands. `50256` is its punctuation mark for "end of message".

In [ ]:
import tiktoken

# Load OpenAI's exact GPT-2 byte-pair encoding tokenizer.
tokenizer = tiktoken.get_encoding("gpt2")

# Confirm the ID of the special <|endoftext|> token.
# Output: [50256]  -- we will reuse 50256 as the PAD token throughout.
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))


---

# 🟥 SECTION 5 — Collate Functions (Batching with Padding)

## 🧠 The problem
Each tokenized example has a **different length** (one might be 50 tokens, another 80). But neural networks need rectangular tensors of uniform shape inside a single batch. We solve this by **padding** all sequences in a batch with a special token (here, `50256`) up to the length of the longest one.

A **collate function** is what `DataLoader` calls to combine a list of single examples into one batched tensor.

We'll see **3 versions** of the collate function — each adding more functionality:
1. **draft 1** — just pad inputs, no targets
2. **draft 2** — also produce shifted targets (next-token prediction)
3. **final** — also mask padding in targets so it doesn't contribute to the loss

---

## 📦 Cell 10 — `custom_collate_draft_1` (inputs only)

### What it builds
For each batch, returns a 2-D tensor of shape `[batch_size, max_seq_length]` where shorter sequences are padded with 50256.

### Line-by-line

<details>
<summary><b><code>def custom_collate_draft_1(batch, pad_token_id=50256, device="cpu"):</code></b></summary>

- `batch` will be a list of pre-tokenized examples (each a list of ints).
- `pad_token_id=50256` — default value if the caller doesn't specify one.
- `device="cpu"` — where to put the resulting tensor (CPU or GPU).

</details>

<details>
<summary><b><code>batch_max_length = max(len(sample)+1 for sample in batch)</code></b></summary>

- **What:** Find the longest sequence length, then add 1.
- **Why the +1?** This is preparation for later versions that need a length-of-N+1 sequence so they can produce both inputs (positions 0..N-1) and targets (positions 1..N) — that is, a shift-by-one sequence for next-token prediction.
- **Generator expression:** `len(s)+1 for s in batch` produces a sequence of values that `max(...)` reduces to one number.

</details>

<details>
<summary><b><code>input_list = []</code></b></summary>

- An empty list to collect each padded example's tensor.

</details>

<details>
<summary><b><code>for item in batch:</code></b></summary>

- Loop through each example.

</details>

<details>
<summary><b><code>new_item = item.copy()</code></b></summary>

- Make a *copy* of the example list. Mutating the original would corrupt the dataset!

</details>

<details>
<summary><b><code>new_item += [pad_token_id]</code></b></summary>

- Append one `50256` to mark end of sequence. List concatenation: `[1,2] + [3] = [1,2,3]`.

</details>

<details>
<summary><b><code>padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))</code></b></summary>

- Adds extra `50256`s until the sequence reaches `batch_max_length`.
- `[pad_token_id] * 5` produces `[50256, 50256, 50256, 50256, 50256]` (list multiplication).

</details>

<details>
<summary><b><code>inputs = torch.tensor(padded[:-1])</code></b></summary>

- 🧠 **PyTorch concept: `torch.tensor`** — a tensor is PyTorch's main data structure (a multi-dimensional array, like NumPy's `ndarray` but with GPU support and autograd).
- `padded[:-1]` slices off the last element (because the +1 above gave us one extra; the last element will live in `targets` next time).
- The result is a 1-D tensor of token IDs.

</details>

<details>
<summary><b><code>input_list.append(inputs)</code></b></summary>

- Add this tensor to the list.

</details>

<details>
<summary><b><code>input_tensor = torch.stack(input_list).to(device)</code></b></summary>

- 🧠 **`torch.stack`** combines a list of equally-shaped 1-D tensors into a single 2-D tensor by stacking them as rows.
  - Input: 3 tensors each of shape `[5]`.
  - Output: 1 tensor of shape `[3, 5]`.
- `.to(device)` moves the tensor to CPU or GPU.

</details>

<details>
<summary><b><code>return input_tensor</code></b></summary>

- Hand back the batched tensor.

</details>

### 📦 Example I/O
With `inputs_1 = [0,1,2,3,4]`, `inputs_2 = [5,6]`, `inputs_3 = [7,8,9]`:
- `batch_max_length = max(6, 3, 4) = 6`
- After +pad and slicing `[:-1]`, each row has 5 tokens.
- Final shape: `[3, 5]`.

In [ ]:
def custom_collate_draft_1(batch, pad_token_id=50256, device="cpu"):
    """First-draft collator: pads each sequence in the batch to the same length
    and returns just the input tensor (no targets yet)."""

    # The +1 ensures we have one extra slot, useful in later drafts for shifted targets
    batch_max_length = max(len(sample) + 1 for sample in batch)
    input_list = []

    for item in batch:
        new_item = item.copy()                # don't mutate the original
        new_item += [pad_token_id]            # append one EOT token

        # Right-pad with PAD tokens up to batch_max_length
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))

        # Drop the last token -> length becomes batch_max_length - 1
        inputs = torch.tensor(padded[:-1])
        input_list.append(inputs)

    # Stack list of 1-D tensors into a single 2-D tensor of shape [batch, seq_len]
    input_tensor = torch.stack(input_list).to(device)
    return input_tensor


## 📦 Cell 11 — Test the draft 1 collator

A tiny manual test using three short fake sequences.

In [ ]:
# Tiny test: fake batch of 3 sequences of unequal length
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (
    inputs_1,
    inputs_2,
    inputs_3,
)
# Expected: shape [3, 5], shorter rows padded with 50256
print(custom_collate_draft_1(batch))


## 📦 Cell 12 — `custom_collate_draft_2` (inputs **and** targets)

### 🧠 New idea: shifted targets for next-token prediction
A language model is trained to **predict the next token at every position**. So if the input is `[A, B, C, D]`, the target is `[B, C, D, E]` — the same sequence shifted left by one. This is *exactly* why we padded with `+1` in draft 1.

### Differences from draft 1
- Returns BOTH `input_tensor` AND `target_tensor`.
- `input_tensor = padded[:-1]` (everything except the last token).
- `target_tensor = padded[1:]`  (everything except the first token).

### 📦 Example I/O
With our same fake batch and `batch_max_length=6`:
- `inputs[0]  = [0, 1, 2, 3, 4]`     ← `padded[:-1]`, length 5
- `targets[0] = [1, 2, 3, 4, 50256]` ← `padded[1:]`, length 5

For the shorter sequences, both inputs and targets get padded with 50256 — but in draft 2 those padding tokens still contribute to the loss (which we'll fix in the final version).

In [ ]:
# Reasoning: a collate function takes individual data samples and combines them
# into one batched tensor by padding to the same length, so the model can process
# them efficiently in parallel. This version also produces SHIFTED targets so the
# model can be trained for next-token prediction.

def custom_collate_draft_2(batch, pad_token_id=50256, device="cpu"):
    batch_max_length = max(len(sample) + 1 for sample in batch)

    inputs_list, targets_list = [], []

    for item in batch:
        # Pad to (batch_max_length) tokens by:
        #  1) appending one EOT token, then 2) right-padding with EOT
        padded_item = item + [pad_token_id]
        padded_item += [pad_token_id] * (batch_max_length - len(padded_item))

        # Inputs:  drop the LAST token  -> positions 0 .. N-1
        # Targets: drop the FIRST token -> positions 1 .. N   (shifted by 1)
        input_item_tensor  = torch.tensor(padded_item[:-1]).to(device)
        target_item_tensor = torch.tensor(padded_item[1:]).to(device)

        inputs_list.append(input_item_tensor)
        targets_list.append(target_item_tensor)

    # Stack into 2-D tensors of shape [batch, seq_len]
    input_tensor  = torch.stack(inputs_list)
    target_tensor = torch.stack(targets_list)
    return input_tensor, target_tensor


## 📦 Cell 13 — Test draft 2

Notice how `targets` is `inputs` shifted left by one position, with the right edge filled in by 50256.

In [ ]:
# Verify: targets should be inputs shifted left by 1 position,
# with the trailing slots padded with 50256.
inputs, targets = custom_collate_draft_2(batch)
print(inputs)
print(targets)


## 📦 Cell 14 — Final `custom_collate_fn` (with **target masking**)

### 🧠 New idea: mask padding in targets with `-100`
PyTorch's cross-entropy loss has a special **`ignore_index`** argument (default `-100`) — any target equal to that value is **completely skipped** during loss computation. This is huge: it means the model isn't penalized for "wrong" predictions on padding positions.

### Strategy
1. Build inputs and targets exactly like draft 2.
2. Find all positions in `targets` equal to the pad token (50256).
3. Keep the **first** such position (which often corresponds to the legitimate end-of-text marker we want the model to learn). Replace **all subsequent** pad-token positions with `-100`.

### Line-by-line of the new bits

<details>
<summary><b><code>mask = targets == pad_token_id</code></b></summary>

- 🧠 **PyTorch broadcast comparison** — produces a Boolean tensor of the same shape, with `True` where `targets[i] == 50256` and `False` elsewhere.

</details>

<details>
<summary><b><code>indices = torch.nonzero(mask).squeeze()</code></b></summary>

- `torch.nonzero(mask)` returns the indices where `mask` is True, as a 2-D tensor (one row per True entry, with that entry's coordinates).
- `.squeeze()` removes size-1 dimensions, giving a 1-D tensor of positions.

</details>

<details>
<summary><b><code>if indices.numel() > 1: targets[indices[1:]] = ignore_index</code></b></summary>

- `indices.numel()` is the total number of elements (the count of True positions).
- If there's more than one pad-token position, we keep the first (`indices[0]`) untouched and set all the rest (`indices[1:]`) to `-100`.

</details>

<details>
<summary><b><code>if allowed_max_length is not None: inputs = inputs[:allowed_max_length] ...</code></b></summary>

- Optional truncation. If the sequence is longer than the model's context window (1024 for GPT-2), trim it.

</details>

### 📦 Example output
For our test batch:
```
inputs:                                     targets:
[[    0,     1,     2,     3,     4],     [[    1,     2,     3,     4, 50256],
 [    5,     6, 50256, 50256, 50256],      [    6, 50256,  -100,  -100,  -100],
 [    7,     8,     9, 50256, 50256]]      [    8,     9, 50256,  -100,  -100]]
```

Notice the **first** padding position in each row is kept as `50256` (so the model still learns to predict end-of-text), but the rest become `-100` (ignored).

In [ ]:
def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100,
                      allowed_max_length=None, device="cpu"):
    """Final, production-grade collator. Same as draft 2, plus:
       - replaces extra padding in TARGETS with -100 so they are ignored by loss
       - optional truncation to allowed_max_length (e.g., 1024 = GPT-2 context)
    """

    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        # 1) Append one pad/EOT token, then right-pad to batch_max_length
        new_item = item.copy() + [pad_token_id]
        padded_item = new_item + [pad_token_id] * (batch_max_length - len(new_item))

        # 2) Build shifted-by-1 input/target sequences for next-token prediction
        inputs  = torch.tensor(padded_item[:-1])
        targets = torch.tensor(padded_item[1:])

        # 3) Mask out EXTRA padding in targets so loss ignores those positions.
        #    Keep the FIRST pad token (acts as legitimate EOT prediction target),
        #    replace all later pad tokens with -100 (PyTorch's ignore_index).
        mask = targets == pad_token_id                 # bool tensor
        indices = torch.nonzero(mask).squeeze()        # positions where True
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # 4) Optional truncation to the model's context window
        if allowed_max_length is not None:
            inputs  = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Stack into [batch, seq_len] tensors and move to the chosen device
    inputs_tensor  = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor


## 📦 Cell 15 — Test the final collator

Verify that extra padding in targets has been replaced by `-100`.

In [ ]:
# Same fake batch as before. The expected output:
#   inputs  - same as draft 2 (shape [3, 5])
#   targets - same as draft 2 BUT with extra 50256s replaced by -100
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)


---

# 🟪 SECTION 6 — Cross-Entropy Loss Demo (why `-100` works)

The next 3 cells **prove experimentally** that `ignore_index=-100` actually causes those positions to be excluded from loss. This isn't strictly necessary for training but is a fantastic sanity check.

## 🧠 PyTorch concept: cross-entropy loss
Given `logits` (raw model scores for each class) and `targets` (the correct class indices), `cross_entropy` computes:
1. Apply softmax to convert logits → probabilities.
2. For each row, take the negative log of the probability assigned to the correct class.
3. Average across rows.

Lower loss = model assigns higher probability to the correct answers.

## 📦 Cell 16 — Compute baseline loss with 2 tokens

### Line-by-line

<details>
<summary><b><code>logits_1 = torch.tensor([[-1.0, 1.0], [-0.5, 1.5]])</code></b></summary>

- A 2x2 tensor. Each row = predictions for one position; each column = one of two possible token classes.
- Shape `[seq_len=2, vocab=2]`.

</details>

<details>
<summary><b><code>targets_1 = torch.tensor([0, 1])</code></b></summary>

- For position 0, correct class is 0. For position 1, correct class is 1.

</details>

<details>
<summary><b><code>loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)</code></b></summary>

- Computes the average cross-entropy over the 2 positions.

</details>

### Output: `tensor(1.1269)`

In [ ]:
# Mini-toy example to understand cross-entropy loss.
# 2 token positions, 2 possible vocab classes each.
logits_1 = torch.tensor(
    [[-1.0, 1.0],   # raw scores for 1st token  (class 0 vs class 1)
     [-0.5, 1.5]]   # raw scores for 2nd token
)
# Correct class indices: position 0 should be class 0, position 1 should be class 1
targets_1 = torch.tensor([0, 1])

# Average cross-entropy across both positions
loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)   # ~1.1269


## 📦 Cell 17 — Add a 3rd identical token; loss changes

Adding a 3rd position with the *same* logits and same correct class **changes** the average loss (because we're now averaging over 3 positions, but they're not all identical to the previous 2).

In [ ]:
# Add a 3rd identical-looking position. As expected, this changes the
# average loss because we now have 3 positions instead of 2.
logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]   # extra row
)
targets_2 = torch.tensor([0, 1, 1])
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)   # ~0.7936  (different from loss_1)


## 📦 Cell 18 — Use `-100` to *ignore* the 3rd position

Now we replace the 3rd target with `-100`. Cross-entropy completely skips it. So we get **the exact same loss as `loss_1`** — proving that `-100` is truly ignored.

### 📌 Key takeaway
This is **why** our `custom_collate_fn` masks padding with `-100` — those positions don't add noise to the gradient signal.

In [ ]:
# Replace the 3rd target with -100 (PyTorch's default ignore_index).
# Cross-entropy will now skip that position entirely.
targets_3 = torch.tensor([0, 1, -100])
loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)                                # equals loss_1
print("loss_1 == loss_3:", loss_1 == loss_3) # tensor(True)


---

# 🟫 SECTION 7 — Device Selection & DataLoaders

## 📦 Cell 19 — Pick GPU if available, else CPU

### Lines
```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
```

### What happens
- 🧠 **`torch.cuda.is_available()`** returns `True` if a CUDA-capable NVIDIA GPU is present and PyTorch can use it.
- The whole expression is a **conditional expression**: pick `"cuda"` if GPU available, else `"cpu"`.
- `torch.device(...)` wraps that string in a Device object that PyTorch operations accept.
- We store it in `device` so we can later say `tensor.to(device)` or `model.to(device)` to put work on the GPU.

### Output (here): `Device: cuda` (a GPU was available)

In [ ]:
# Pick GPU if available, otherwise fall back to CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 📦 Cell 20 — Pre-bind device & max length to the collate function

### Lines
```python
from functools import partial
customized_collate_fn = partial(custom_collate_fn, device=device, allowed_max_length=1024)
```

### 🧠 Concept: `functools.partial`
`partial(f, x=5)` returns a NEW function which is `f` with the keyword `x=5` already filled in. So later calls only need to provide the *remaining* arguments.

### Why we need it
PyTorch's `DataLoader` will call our collate function with **only one argument**: the batch. But our `custom_collate_fn` needs `device` and `allowed_max_length` too. `partial` glues those in advance.

### 🧪 Real-world analogy
Like setting your microwave's default time to 30 seconds — every time you press start, it automatically uses 30 seconds without asking.

In [ ]:
from functools import partial

# DataLoader passes just one argument (the batch) to the collate fn.
# We pre-fill the other two parameters so DataLoader's call works correctly.
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024,   # GPT-2's max context window
)


## 📦 Cell 21 — Build train, validation, and test DataLoaders

### 🧠 PyTorch concept: `DataLoader`
A `DataLoader` wraps a `Dataset` and provides:
- **Batching** — group N examples together.
- **Shuffling** — randomize order each epoch.
- **Parallel loading** — use multiple worker processes (set with `num_workers`).
- **Custom collation** — uses your `collate_fn` to combine examples into a tensor.

### Line-by-line

<details>
<summary><b><code>from torch.utils.data import DataLoader</code></b></summary>

- Import the class.

</details>

<details>
<summary><b><code>num_workers = 0</code></b></summary>

- Number of background processes that load data in parallel. `0` = main process only (simpler, no multiprocessing overhead).

</details>

<details>
<summary><b><code>batch_size = 8</code></b></summary>

- 8 examples per batch — a common small batch size that fits in modest GPUs.

</details>

<details>
<summary><b><code>torch.manual_seed(123)</code></b></summary>

- Seeds PyTorch's random number generator. Makes shuffling reproducible: every run gets the same shuffle order.

</details>

<details>
<summary><b><code>train_dataset = InstructionDataset(train_data, tokenizer)</code></b></summary>

- Wraps our raw training list into the Dataset class — pre-tokenizes everything inside `__init__`.

</details>

<details>
<summary><b><code>train_loader = DataLoader(train_dataset, batch_size=8, collate_fn=customized_collate_fn, shuffle=True, drop_last=True, num_workers=0)</code></b></summary>

- `shuffle=True` — randomize order each epoch (good for training).
- `drop_last=True` — discard the final partial batch if it has < 8 examples (keeps shapes consistent).

</details>

<details>
<summary><b><code>val_loader = DataLoader(...)</code></b><br>
<code>test_loader = DataLoader(...)</code></summary>

- Same pattern, but `shuffle=False` and `drop_last=False` because we want deterministic evaluation over **every** example.

</details>

In [ ]:
from torch.utils.data import DataLoader

num_workers = 0      # Single-process data loading (simplest)
batch_size  = 8      # 8 examples per batch

torch.manual_seed(123)  # Reproducible shuffling

# ---- Training loader (SHUFFLE + drop_last for steady batch shapes) ----
train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers,
)

# ---- Validation loader (no shuffle, no drop) ----
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)

# ---- Test loader (no shuffle, no drop) ----
test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)


## 📦 Cell 22 — Verify batch shapes

### Lines
```python
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)
```

### What this does
Iterates through the entire training loader once, printing each batch's shape.

### Why shapes differ between batches
Each batch has shape `[8, max_len_in_THIS_batch]`. The longest sequence in batch A might be 61 tokens, in batch B might be 76 tokens. So shapes vary like:
```
torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 73]) torch.Size([8, 73])
...
```
This is **dynamic padding** — we only pad each batch up to *that* batch's max, not the global max. Saves a lot of compute.

### 📌 Important: `inputs.shape == targets.shape` always
Both have the same `[batch, seq_len]` shape because targets are just inputs-shifted-by-one with masking applied.

In [ ]:
# Sanity-check: walk through the train loader and print each batch's shape.
# We should see shape [8, varying] -- the seq_len differs per batch (dynamic padding).
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)


---

# 🟦 SECTION 8 — Load a Pre-trained GPT-2 Model

## 📦 Cell 23 — Download GPT-2 weights and assemble the model

This cell uses helper modules (from the *LLMs from Scratch* book) to:
1. Download GPT-2 medium (355M parameter) weights from OpenAI's servers.
2. Construct a `GPTModel` PyTorch module with the right architecture.
3. Copy the downloaded weights into our model.

### Line-by-line

<details>
<summary><b>The 3 helper imports</b></summary>

- `download_and_load_gpt2` — downloads & loads OpenAI's pre-trained GPT-2 weights.
- `GPTModel` — the PyTorch model class (defines architecture).
- `load_weights_into_gpt` — copies the OpenAI weight tensors into our model.

</details>

<details>
<summary><b><code>BASE_CONFIG = {...}</code></b></summary>

- A dict of architecture-independent hyperparameters:
  - `vocab_size = 50257` — total tokens GPT-2 knows (50k regular + 257 special).
  - `context_length = 1024` — max input length.
  - `drop_rate = 0.0` — dropout probability (0 means no dropout).
  - `qkv_bias = True` — whether the attention's Q/K/V projections include biases (GPT-2 does).

</details>

<details>
<summary><b><code>model_configs = {...}</code></b></summary>

- A lookup table of architecture-dependent params for each GPT-2 size.
- `emb_dim` = embedding dimension; `n_layers` = transformer blocks; `n_heads` = attention heads.

</details>

<details>
<summary><b><code>CHOOSE_MODEL = "gpt2-medium (355M)"</code></b></summary>

- Pick the medium model (355M parameters). Big enough to be useful, small enough to fine-tune on a single GPU.

</details>

<details>
<summary><b><code>BASE_CONFIG.update(model_configs[CHOOSE_MODEL])</code></b></summary>

- Adds the size-specific keys (`emb_dim`, `n_layers`, `n_heads`) to `BASE_CONFIG` in place.

</details>

<details>
<summary><b><code>model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")</code></b></summary>

- String surgery to extract `"355M"` from `"gpt2-medium (355M)"`:
  - `.split(" ")` → `["gpt2-medium", "(355M)"]`
  - `[-1]` → `"(355M)"`
  - `.lstrip("(")` strips opening paren → `"355M)"`
  - `.rstrip(")")` strips closing paren → `"355M"`

</details>

<details>
<summary><b><code>settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")</code></b></summary>

- Downloads weights into a folder called `gpt2/` and returns:
  - `settings` — metadata (also describing architecture).
  - `params` — the actual numerical weights.

</details>

<details>
<summary><b><code>model = GPTModel(BASE_CONFIG)</code></b></summary>

- 🧠 **Instantiating a PyTorch model**: This creates a `GPTModel` object with **randomly initialized weights** matching the architecture spec.

</details>

<details>
<summary><b><code>load_weights_into_gpt(model, params)</code></b></summary>

- Copies the downloaded OpenAI weights INTO our model, replacing the random values.

</details>

<details>
<summary><b><code>model.eval();</code></b></summary>

- 🧠 **`model.eval()`** puts the model in **evaluation mode**: turns off dropout and uses running statistics in batchnorm. We use this when generating text or computing loss without training.
- The trailing `;` just suppresses the printout of the model architecture in Jupyter.

</details>

In [ ]:
from gpt_download import download_and_load_gpt2
from gpt_model import GPTModel
from load_weights_into_gpt import load_weights_into_gpt

# Architecture parameters that don't depend on the chosen model size
BASE_CONFIG = {
    "vocab_size":     50257,   # GPT-2 vocabulary size
    "context_length": 1024,    # Maximum sequence length the model can handle
    "drop_rate":      0.0,     # Dropout probability (0 = none for inference)
    "qkv_bias":       True,    # GPT-2 uses bias in QKV projections
}

# Size-specific parameters — pick one of these to use
model_configs = {
    "gpt2-small (124M)":   {"emb_dim": 768,  "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)":  {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)":   {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)":     {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"           # 355M params: a sweet spot
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])  # Merge size-specific values in

# Extract the bare size string ("355M") from "gpt2-medium (355M)"
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")

# Download (or load from disk) OpenAI's pre-trained GPT-2 medium weights
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

# Build a fresh GPTModel with the right architecture (random weights)
model = GPTModel(BASE_CONFIG)
# Then overwrite the random weights with the pre-trained OpenAI weights
load_weights_into_gpt(model, params)

# Put model in evaluation mode (disables dropout). The ';' suppresses Jupyter's
# auto-print of the entire model architecture.
model.eval();


## 📦 Cell 24 — Try generating with the un-fine-tuned model

Now we let the **pre-trained-but-not-yet-fine-tuned** model attempt to follow an instruction. This gives us a baseline — typically it fails or just rambles, because it hasn't learned the Alpaca format yet.

### Line-by-line

<details>
<summary><b><code>from generate import generate</code><br>
<code>from text_to_id_to_text import text_to_token_ids, token_ids_to_text</code></b></summary>

- Helper utilities: `generate` runs the model autoregressively to produce new tokens; the others convert between strings and ID tensors.

</details>

<details>
<summary><b><code>torch.manual_seed(123)</code></b></summary>

- Seed for reproducibility (in case generation uses randomness).

</details>

<details>
<summary><b><code>input_text = format_input(val_data[0])</code></b></summary>

- Take the FIRST validation example and format it as an Alpaca prompt (without the response).

</details>

<details>
<summary><b><code>print(input_text)</code></b></summary>

- Show the prompt we're feeding in.

</details>

<details>
<summary><b><code>token_ids = generate(...)</code></b></summary>

- Calls the generation helper with these arguments:
  - `model=model.to(device)` — move model to GPU first.
  - `idx=text_to_token_ids(input_text, tokenizer).to(device)` — encode the prompt as a tensor on GPU.
  - `max_new_tokens=35` — generate up to 35 tokens.
  - `context_size=1024` — max context window.
  - `eos_id=50256` — stop early if 50256 (end-of-text) is generated.

</details>

<details>
<summary><b><code>generated_text = token_ids_to_text(token_ids, tokenizer)</code></b></summary>

- Decode the IDs back into a human-readable string. (Note: this cell doesn't print it; the next cell does.)

</details>

In [ ]:
from generate import generate
from text_to_id_to_text import text_to_token_ids, token_ids_to_text

torch.manual_seed(123)

# Take the FIRST validation example and format only its prompt portion.
input_text = format_input(val_data[0])
print(input_text)

# Generate continuation. The model will try (probably poorly, since it hasn't
# been fine-tuned on instruction format yet) to produce a response.
token_ids = generate(
    model=model.to(device),
    idx=text_to_token_ids(input_text, tokenizer).to(device),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256,    # stop early if the EOT token is sampled
)

# Decode token IDs back to text (next cell will print just the response part)
generated_text = token_ids_to_text(token_ids, tokenizer)


## 📦 Cell 25 — Extract just the generated response

### Lines
```python
response_text = generated_text[len(input_text):].strip()
print(response_text)
```

### What it does
- `generated_text` is the **prompt + generation** concatenated. To see only what the model produced, we slice off the leading `len(input_text)` characters.
- `.strip()` removes leading/trailing whitespace.

### What you'll see
The model probably outputs something incoherent or off-topic — that's expected, and it's exactly the gap fine-tuning will close.

In [ ]:
# Slice off the prompt portion and print just the model's continuation.
# Expect this to be poor BEFORE fine-tuning -- that's our motivation to train.
response_text = generated_text[len(input_text):].strip()
print(response_text)


---

# 🟪 SECTION 9 — Measure Initial Loss (Before Fine-tuning)

## 📦 Cell 26 — Compute baseline train/val losses

We measure how poorly the un-fine-tuned model performs on instruction-following data, so we'll know later how much fine-tuning helped.

### Line-by-line

<details>
<summary><b><code>from train_model_simple import train_model_simple</code><br>
<code>from calc_loss_loader import calc_loss_loader</code></b></summary>

- Helpers: `train_model_simple` will run the actual training loop (next cell). `calc_loss_loader` computes average loss over a DataLoader.

</details>

<details>
<summary><b><code>model.to(device)</code></b></summary>

- Make sure the model is on GPU.

</details>

<details>
<summary><b><code>torch.manual_seed(123)</code></b></summary>

- Reproducibility seed.

</details>

<details>
<summary><b><code>with torch.no_grad():</code></b></summary>

- 🧠 **`torch.no_grad()`** is a **context manager** that tells PyTorch: "Don't track gradients for any operations in this block."
- **Why:** Gradient tracking uses extra memory and time. We don't need gradients here because we're just measuring loss (no backprop).
- All operations inside this `with` block run faster and use less memory.

</details>

<details>
<summary><b><code>train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)</code></b><br>
<code>val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)</code></summary>

- Computes mean loss over only **5 batches** of each loader (just for a quick estimate; full evaluation would be slower).

</details>

### 🧪 Real-world analogy
Like measuring how high a student scores on a test BEFORE studying, so later you can see how much they improved.

In [ ]:
from train_model_simple import train_model_simple
from calc_loss_loader  import calc_loss_loader

model.to(device)
torch.manual_seed(123)

# torch.no_grad() = "do these operations without tracking gradients"
# (saves memory + time when we just want to measure loss, not train).
with torch.no_grad():
    # Average loss over 5 batches of training data
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    # Average loss over 5 batches of validation data
    val_loss   = calc_loss_loader(val_loader,   model, device, num_batches=5)

print("Training loss:",   train_loss)
print("Validation loss:", val_loss)


---

# 🟧 SECTION 10 — The Fine-tuning Loop

## 📦 Cell 27 — Actually train the model

This is the heart of the notebook. We set up an optimizer and call `train_model_simple` to run the gradient-descent loop.

### Line-by-line

<details>
<summary><b><code>import time</code><br><code>start_time = time.time()</code></b></summary>

- Record the wall-clock time before training so we can measure how long it took.

</details>

<details>
<summary><b><code>torch.manual_seed(123)</code></b></summary>

- Reproducible random init (for things like dropout if any, weight init in any added layers).

</details>

<details>
<summary><b><code>optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)</code></b></summary>

- 🧠 **PyTorch concept: optimizer** — the algorithm that updates model weights using gradients.
- **`AdamW`** = Adam with **decoupled weight decay**. The standard choice for fine-tuning transformers.
- `model.parameters()` returns an iterator over all trainable weight tensors in the model — that's what AdamW will update.
- `lr=0.00005` (5e-5) — learning rate. Small because we're fine-tuning (don't want to disturb the pre-trained weights too much).
- `weight_decay=0.1` — regularization that gently pulls weights toward zero, helping prevent overfitting.

</details>

<details>
<summary><b><code>num_epochs = 2</code></b></summary>

- 🧠 **Epoch** = one full pass through the entire training set. We train for 2 epochs.

</details>

<details>
<summary><b><code>train_losses, val_losses, tokens_seen = train_model_simple(...)</code></b></summary>

- The big call. Runs the actual training loop. Returns:
  - `train_losses` — list of training losses recorded during training.
  - `val_losses` — list of validation losses.
  - `tokens_seen` — list of cumulative token counts (for plotting).
- Arguments:
  - `model, train_loader, val_loader, optimizer, device` — the basics.
  - `num_epochs=2` — train for 2 epochs.
  - `eval_freq=5` — measure validation loss every 5 training steps.
  - `eval_iter=5` — average over 5 batches when measuring validation loss.
  - `start_context=format_input(val_data[0])` — periodically generate text from this prompt to see qualitative progress.
  - `tokenizer=tokenizer` — needed for the periodic generation.

</details>

<details>
<summary><b><code>end_time = time.time()</code><br>
<code>execution_time_minutes = (end_time - start_time) / 60</code><br>
<code>print(f"Training completed in {execution_time_minutes:.2f} minutes.")</code></b></summary>

- Compute and pretty-print the elapsed time. The `:.2f` inside the f-string formats the float to 2 decimal places.

</details>

### 🧪 Real-world analogy
Setting up driving lessons: AdamW is the driving instructor, learning rate is how aggressively you correct mistakes, epochs is how many times you drive around the practice track.

In [ ]:
import time
start_time = time.time()

torch.manual_seed(123)

# AdamW: Adam optimizer + decoupled weight decay (standard for transformer fine-tuning).
#  - lr=5e-5: small learning rate, typical for fine-tuning so we don't blow up pre-trained weights
#  - weight_decay=0.1: regularization to discourage huge weights
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)

# Number of full passes through the training set
num_epochs = 2

# Run the training loop. Returns recorded losses + cumulative token counts for plotting.
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs,
    eval_freq=5,                                # measure val loss every 5 steps
    eval_iter=5,                                # avg over 5 batches per measurement
    start_context=format_input(val_data[0]),    # qualitative sample prompt
    tokenizer=tokenizer,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")


---

# 🟩 SECTION 11 — Visualize Training Progress

## 📦 Cell 28 — Plot training & validation losses

### Line-by-line

<details>
<summary><b><code>from plot_losses import plot_losses</code></b></summary>

- Imports a helper that draws a 2-axis matplotlib plot (epochs on x-axis, with a secondary x-axis showing tokens seen).

</details>

<details>
<summary><b><code>epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))</code></b></summary>

- 🧠 **`torch.linspace(start, end, n)`** generates `n` equally-spaced values from `start` to `end` inclusive.
- We need this because we recorded `len(train_losses)` loss values during training and want to map each one to a fractional epoch on the x-axis.

</details>

<details>
<summary><b><code>plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)</code></b></summary>

- Plot it! You'll see two curves descending — train loss going down sharply, val loss following (hopefully without diverging upward, which would indicate overfitting).

</details>

### 📌 What a healthy plot looks like
- Both curves drop quickly at first, then flatten.
- Training loss usually drops faster than validation loss.
- If validation loss starts going *up* while training loss keeps dropping → overfitting. Stop earlier or regularize more.

In [ ]:
from plot_losses import plot_losses

# Make x-axis values: equally-spaced "epoch fractions" matching # of recorded losses
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))

# Draw the loss curves vs both epochs (bottom axis) and tokens_seen (top axis)
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)


---

# 🧩 FINAL — Summary, Goal, & Key Concepts

## 🧾 Plain-English summary of the file

We took 1,100 instruction-response examples (e.g., "Identify the spelling of *Ocassion*" → "Occasion"), wrapped each in the **Alpaca prompt format**, split them into 935 train / 55 validation / 110 test. We then loaded a pre-trained GPT-2 medium (355M parameters) model, measured its loss on the data **before** fine-tuning (poor), and then **fine-tuned** it for 2 epochs using AdamW. Finally we plotted how the loss decreased over training.

## 🎯 Main goal of this code
**Teach a pre-trained GPT-2 model to follow human instructions in the Alpaca format**, by continuing to train it on a small instruction-tuned dataset.

## 🔄 How all parts connect
1. `download_and_load_file` → raw JSON list of dicts.
2. `format_input` → wraps each dict in the Alpaca prompt template.
3. `InstructionDataset` → pre-tokenizes every example using the GPT-2 tokenizer.
4. `custom_collate_fn` → batches examples with padding + masked targets.
5. `DataLoader` → iterates batches, shuffled & on the right device.
6. `GPTModel` + `load_weights_into_gpt` → instantiate model with OpenAI's pre-trained weights.
7. `calc_loss_loader` → measure baseline loss.
8. `AdamW` + `train_model_simple` → actually fine-tune.
9. `plot_losses` → visualize.

## 🧠 Key PyTorch concepts to remember

| Concept | What it is | Where seen |
|---|---|---|
| **Tensor** | Multi-dim array w/ GPU + autograd | Everywhere |
| **`Dataset`** | Class with `__len__` + `__getitem__` | `InstructionDataset` |
| **`DataLoader`** | Iterates batches, handles shuffling/parallelism | Cell 21 |
| **`collate_fn`** | Function that combines samples → batch tensor | `custom_collate_fn` |
| **`torch.tensor`** | Wrap a Python list as a tensor | Throughout |
| **`torch.stack`** | Combine list of equal-shape tensors → 1 bigger tensor | Collate fns |
| **`tensor.to(device)`** | Move tensor to CPU/GPU | Throughout |
| **`torch.no_grad()`** | Disable gradient tracking (eval mode) | Cell 26 |
| **`model.eval()`** | Put model in inference mode | Cell 23 |
| **Cross-entropy + `ignore_index=-100`** | Skip padding positions in loss | Cells 16-18, collate fn |
| **Optimizer (AdamW)** | Updates model weights from gradients | Cell 27 |
| **Epoch** | One full pass over training data | Cell 27 |
| **`torch.linspace`** | Equally-spaced values | Cell 28 |
| **`torch.manual_seed`** | Reproducible randomness | Throughout |

## 🔑 The single most important idea
**Padding tokens shouldn't influence the loss.** That's why we use `ignore_index=-100` — it lets us pad sequences to a uniform length for efficient batching while keeping the gradient signal clean. This trick is fundamental to almost all sequence-model training in PyTorch.
